# Time Series Forecasting with AutoGluon

**MIDAS AI in Research Handbook — Time Series Forecasting**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/autogluon_timeseries_demo.ipynb)

---

This notebook walks through a complete time series forecasting workflow using AutoGluon's `TimeSeriesPredictor`. By the end, you will have trained a multi-model forecast, read the leaderboard, interpreted uncertainty intervals, and compared a classical model against a modern foundation model — all with under 30 lines of code.

**Dataset:** A 20-series subset of the M4 Monthly forecasting benchmark. Each series is a monthly time series of real-world observations spanning demographic, economic, and industrial domains. The task is to forecast the next 12 months.

**What you need:** A Google account to run in Colab. No local installation required.

## Step 1 — Install AutoGluon

This takes about 3–5 minutes on a fresh Colab session. Run it once and then proceed.

In [ ]:
!pip install -q "autogluon.timeseries" 2>&1 | tail -5
print("Installation complete.")

## Step 2 — Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

print("AutoGluon imported successfully.")

## Step 3 — Load the Dataset

We load a 20-series subset of the M4 Monthly dataset directly from a public URL. The M4 competition is a standard forecasting benchmark with series from multiple real-world domains.

The data arrives in **wide format**: one row per series, one column per time point. This is how many researchers store longitudinal data, so the first thing we do is reshape it into the **long format** that AutoGluon requires.

In [ ]:
# Load M4 monthly subset (wide format)
url = "https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/Train/Monthly-train.csv"
df_wide = pd.read_csv(url, nrows=20)

print(f"Wide format shape: {df_wide.shape}")
print("First few columns:", df_wide.columns[:5].tolist())
df_wide.iloc[:3, :6]

## Step 4 — Reshape to Long Format

AutoGluon requires **long format**: each row is one observation of one series at one time point, with columns for `item_id`, `timestamp`, and `target`.

Most longitudinal research data starts in wide format, so this reshaping step is something you will encounter often. We use `pandas.melt()` to do it.

In [ ]:
# The V1, V2, ... columns are monthly observations. 
# We'll create timestamps starting from Jan 2000 for illustration.
n_series, n_time = df_wide.shape[0], df_wide.shape[1] - 1  # minus the 'V1' id col

# Use the row index as item_id (M1, M2, ...)
df_wide.insert(0, "item_id", ["M" + str(i+1) for i in range(n_series)])
df_wide = df_wide.drop(columns=["V1"], errors="ignore")

# Rename value columns to sequential numbers
value_cols = [c for c in df_wide.columns if c != "item_id"]

# Melt to long format
df_long = df_wide.melt(id_vars="item_id", var_name="period", value_name="target")
df_long = df_long.dropna(subset=["target"])

# Create timestamps: monthly starting 2000-01-01
df_long["period_num"] = df_long["period"].str.replace("V", "", regex=False).astype(int) - 1
start = pd.Timestamp("2000-01-01")
df_long["timestamp"] = df_long["period_num"].apply(lambda n: start + pd.DateOffset(months=n))
df_long = df_long[["item_id", "timestamp", "target"]].sort_values(["item_id", "timestamp"]).reset_index(drop=True)

print(f"Long format shape: {df_long.shape}")
print(f"Series count: {df_long['item_id'].nunique()}")
print(f"Time range: {df_long['timestamp'].min().date()} to {df_long['timestamp'].max().date()}")
df_long.head(8)

## Step 5 — Visualize the Series

Before modeling, it helps to look at the data. Real-world time series often show trend, seasonality, or both. Understanding the visual pattern helps you interpret the leaderboard later.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
axes = axes.flatten()

for i, series_id in enumerate(["M1", "M2", "M3", "M4", "M5", "M6"]):
    ax = axes[i]
    s = df_long[df_long["item_id"] == series_id]
    ax.plot(s["timestamp"], s["target"], linewidth=1.5)
    ax.set_title(series_id, fontsize=11)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.tick_params(axis="x", labelsize=8)
    ax.tick_params(axis="y", labelsize=8)

plt.suptitle("Six sample series from the M4 Monthly dataset", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("series_preview.png", dpi=120, bbox_inches="tight")
plt.show()
print("Notice how series differ in scale, trend, and seasonality.")

## Step 6 — Convert to TimeSeriesDataFrame and Split

AutoGluon works with a `TimeSeriesDataFrame`, which is essentially a pandas DataFrame with some additional time series-aware structure. We convert our long-format DataFrame in one line.

Then we split into train and test. The split **must be temporal**: we hold out the last 12 months of each series as the test set, and train on everything before that.

In [ ]:
PREDICTION_LENGTH = 12  # forecast 12 months ahead

# Convert to AutoGluon's format
ts_df = TimeSeriesDataFrame.from_data_frame(
    df_long,
    id_column="item_id",
    timestamp_column="timestamp"
)

# Hold out the last prediction_length time steps of each series for testing
train_data = ts_df.slice_by_timestep(None, -PREDICTION_LENGTH)
test_data  = ts_df  # full series; we'll compare predictions against the held-out portion

print(f"Training data: {len(train_data)} rows")
print(f"Each series has roughly {len(train_data) // ts_df['item_id'].nunique() if hasattr(ts_df, 'item_id') else '?'} training observations")
print(f"We will forecast {PREDICTION_LENGTH} months ahead for each series.")

## Step 7 — Train the Predictor

This is the core step. We create a `TimeSeriesPredictor` and call `fit()`.

A few things to notice about the setup:
- `prediction_length=12` tells AutoGluon the forecast horizon.
- `eval_metric="MASE"` is the metric AutoGluon will use to rank models and build the ensemble. We explain MASE in the next cell.
- `time_limit=120` gives AutoGluon two minutes. That is enough for a first feasibility pass.
- `presets="medium_quality"` balances speed and model coverage.

AutoGluon will print a training log showing which models it tries and their validation scores. This is useful to read: you will see the statistical models finish in seconds and the deeper models take longer.

In [ ]:
predictor = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    target="target",
    eval_metric="MASE",
    path="autogluon_ts_model"
).fit(
    train_data,
    time_limit=120,
    presets="medium_quality",
    verbosity=2
)

## Step 8 — Understanding MASE

Before reading the leaderboard, let's understand what the metric means.

**MASE (Mean Absolute Scaled Error)** compares your model's error against a naive seasonal baseline. The baseline simply repeats the last observed seasonal value as its forecast. A MASE of:
- **< 1.0**: your model beats the naive baseline — there is learnable signal in the data
- **= 1.0**: your model is no better than the baseline
- **> 1.0**: the naive baseline is actually better — a warning sign worth investigating

Because MASE is scale-free (it normalizes by the baseline error), it is comparable across all 20 series even though they have very different scales. This is why it is the standard metric in forecasting benchmarks like M4.

AutoGluon reports it as a **negative number** in the leaderboard (so that higher is always better, consistent with its convention across all metrics). A leaderboard score of `-0.82` corresponds to a MASE of `0.82`.

## Step 9 — Inspect the Leaderboard

The leaderboard shows every model that was trained, ranked by validation MASE. Higher is better (remember: the scores are negative, so something like `-0.75` is better than `-0.90`).

In [ ]:
leaderboard = predictor.leaderboard(test_data, silent=True)
print(leaderboard[["model", "score_test", "score_val", "fit_time_marginal"]].to_string(index=False))

In [ ]:
# Visualize the leaderboard as a bar chart
lb_plot = leaderboard.sort_values("score_test")
colors = ["steelblue" if "Ensemble" not in m else "darkorange" for m in lb_plot["model"]]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(lb_plot["model"], lb_plot["score_test"], color=colors)
ax.set_xlabel("Test MASE (higher = better; scores are negated)", fontsize=10)
ax.set_title("AutoGluon Time Series Leaderboard", fontsize=12)
ax.axvline(x=0, color="gray", linestyle="--", linewidth=0.8)

# Add a legend note
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="steelblue", label="Individual model"),
                   Patch(facecolor="darkorange", label="Ensemble")]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig("leaderboard.png", dpi=120, bbox_inches="tight")
plt.show()

## Step 10 — Generate and Visualize Forecasts

`predict()` returns a DataFrame with quantile forecasts for each series. By default you get the 10th, 50th, and 90th percentiles.

- The **50th percentile** is the point forecast (your best single estimate).
- The **10th–90th interval** is an 80% prediction interval. In a well-calibrated model, the true future value should fall inside this band about 80% of the time.

The width of the interval tells you how confident the model is. Narrow bands mean the model sees a clear pattern; wide bands signal high uncertainty.

In [ ]:
predictions = predictor.predict(train_data)
print("Prediction output shape:", predictions.shape)
print("Columns (quantile levels):", predictions.columns.tolist())
predictions.head(6)

In [ ]:
# Plot forecast with uncertainty interval for three series
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, series_id in zip(axes, ["M1", "M2", "M3"]):
    # Historical
    hist = train_data.loc[series_id] if series_id in train_data.index.get_level_values(0) else None
    # Actual held-out
    actual = test_data.loc[series_id] if series_id in test_data.index.get_level_values(0) else None
    # Forecast
    pred = predictions.loc[series_id] if series_id in predictions.index.get_level_values(0) else None

    if hist is None or pred is None:
        ax.set_title(f"{series_id} (no data)")
        continue

    hist_tail = hist.tail(36)
    ax.plot(hist_tail.index, hist_tail["target"], color="steelblue", linewidth=1.5, label="History")

    if actual is not None:
        actual_tail = actual.tail(PREDICTION_LENGTH)
        ax.plot(actual_tail.index, actual_tail["target"], color="black",
                linewidth=1.5, linestyle="--", label="Actual")

    ax.plot(pred.index, pred["0.5"], color="darkorange", linewidth=2, label="Forecast (median)")
    ax.fill_between(pred.index, pred["0.1"], pred["0.9"],
                    alpha=0.25, color="darkorange", label="80% interval")

    ax.set_title(series_id, fontsize=11)
    ax.tick_params(axis="x", labelsize=8, rotation=20)
    ax.legend(fontsize=7)

plt.suptitle("Forecast vs. actuals for three series\n(orange = forecast, black dashed = actual, shaded = 80% interval)",
             fontsize=11, y=1.03)
plt.tight_layout()
plt.savefig("forecast_plot.png", dpi=120, bbox_inches="tight")
plt.show()

## Step 11 — Evaluate on the Test Set

`predictor.evaluate()` reports performance of the best model on the held-out test portion, across multiple metrics at once.

In [ ]:
scores = predictor.evaluate(test_data)
print("\nTest set evaluation (best model):")
for metric, val in sorted(scores.items()):
    # Convert negated metrics back to positive for display
    display_val = abs(val) if val < 0 else val
    sign = "" if val >= 0 else "lower is better → "
    print(f"  {metric:<30} {sign}{display_val:.4f}")

## Step 12 — Zero-Shot Forecasting with Chronos-2

Chronos-2 is a foundation model pretrained on hundreds of millions of real and synthetic time series. Unlike every other model in the leaderboard, it does **not train on your data at all**. It generates forecasts purely from what it learned during pretraining — this is called zero-shot forecasting.

This is worth trying separately because it behaves differently from the other models:
- It returns almost instantly (no fitting required)
- It can be surprisingly competitive on small or irregular datasets
- It provides a useful baseline: if Chronos matches your trained ensemble, your dataset may not have patterns that benefit from training

The `"chronos2_small"` preset uses the smaller Chronos-2 model, which runs well on CPU.

In [ ]:
predictor_chronos = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    target="target",
    eval_metric="MASE",
    path="autogluon_ts_chronos"
).fit(
    train_data,
    presets="chronos2_small",  # zero-shot: fit() returns almost instantly
    verbosity=1
)

chronos_scores = predictor_chronos.evaluate(test_data)
main_mase = abs(scores.get("MASE", scores.get("-MASE", list(scores.values())[0])))
chron_mase = abs(chronos_scores.get("MASE", chronos_scores.get("-MASE", list(chronos_scores.values())[0])))

print(f"\nMASE comparison:")
print(f"  Full ensemble (trained): {main_mase:.4f}")
print(f"  Chronos-2 zero-shot:     {chron_mase:.4f}")
print()
if chron_mase < main_mase:
    print("Chronos-2 wins here — the zero-shot model is competitive.")
    print("This often happens when training data is limited and Chronos can rely on pretraining.")
else:
    print("The trained ensemble wins — the extra training time paid off.")
    print("This is typical when series have strong domain-specific patterns.")

## Step 13 — Reshaping Your Own Data (Wide to Long)

If your data starts in wide format (a common structure in social science and clinical research), here is the general pattern for converting it. This works whether your columns are labeled as weeks, dates, visit numbers, or anything else.

In [ ]:
# --- Template: Reshaping wide longitudinal data to long format ---
# Replace column names and values with your own data

import pandas as pd

# Example: 3 subjects measured at 5 weekly time points
example_wide = pd.DataFrame({
    "subject_id": ["subj_01", "subj_02", "subj_03"],
    "week_1":  [72, 80, 65],
    "week_2":  [74, 78, 67],
    "week_3":  [70, 82, 66],
    "week_4":  [73, 79, 68],
    "week_5":  [75, 81, 70],
})

# Step 1: Melt to long format
time_cols = ["week_1", "week_2", "week_3", "week_4", "week_5"]
long = example_wide.melt(
    id_vars="subject_id",
    value_vars=time_cols,
    var_name="week",
    value_name="target"
)

# Step 2: Create actual timestamps (adjust start date and frequency for your data)
week_map = {col: pd.Timestamp("2023-01-01") + pd.DateOffset(weeks=i)
            for i, col in enumerate(time_cols)}
long["timestamp"] = long["week"].map(week_map)

# Step 3: Rename item column and clean up
long = long.rename(columns={"subject_id": "item_id"})
long = long[["item_id", "timestamp", "target"]].sort_values(["item_id", "timestamp"]).reset_index(drop=True)

print("Reshaped long format:")
print(long.to_string(index=False))

---

## Hands-On Exercises

These exercises build on what you have done above. Try them in order — each one reveals something meaningful about how `TimeSeriesPredictor` behaves.

**Exercise 1 — Change the forecast horizon**
Retrain with `prediction_length=6` instead of 12. Does the leaderboard ranking change? Does MASE improve? What does that tell you about which models handle shorter horizons better?

**Exercise 2 — Shorten the time limit**
Try `time_limit=30`. How many models still get trained? Which ones get dropped first? This is a useful exercise for understanding the trade-off between speed and model coverage.

**Exercise 3 — Try a different preset**
Run with `presets="fast_training"`. Compare the leaderboard to your `"medium_quality"` run. Is the MASE meaningfully worse, or roughly similar?

**Exercise 4 — Look at calibration**
Plot the 80% prediction interval for one series and count how many of the 12 actual values fall inside it. You expect about 10 out of 12 (80%). If far fewer fall inside, the model is overconfident. If nearly all of them fall inside, the bands may be too wide.

**Exercise 5 — Adapt the wide-to-long template for your own data**
If you have a longitudinal dataset of your own, use the template in Step 13 to reshape it and run `TimeSeriesPredictor` on it. What does the SeasonalNaive score tell you?


---

## Common Questions

These came up in workshops and are worth knowing before you run this on your own data.


### Why does the training log show negative MASE scores?

AutoGluon uses a "higher is always better" convention for all metrics, including error metrics. This means error metrics like MASE are multiplied by -1 so they fit the same ranking logic. A score of `-0.82` means a MASE of `0.82`. To recover the actual error, take the absolute value.


### SeasonalNaive beat my trained models. Is something wrong?

Not necessarily. SeasonalNaive repeating the last seasonal cycle is sometimes hard to beat, especially on short series where the seasonal pattern is strong and stable. If SeasonalNaive wins, it suggests either: (a) the series are strongly seasonal with little additional structure, or (b) your training series are too short for the global models to learn from. Both are useful things to know.


### My timestamps are not regular — I have some missing weeks. What should I do?

AutoGluon infers the frequency from your timestamps. If some observations are missing, the easiest fix is to create a complete regular date range and join your data to it (leaving missing values as `NaN`). AutoGluon will handle the `NaN` values internally. Here is a quick pattern:

```python
# Create a complete regular index
full_idx = pd.date_range(start=df["timestamp"].min(),
                         end=df["timestamp"].max(), freq="W")
df_complete = df.set_index("timestamp").reindex(full_idx).rename_axis("timestamp").reset_index()
# NaN rows are now explicit; AutoGluon will handle them
```


### How do I add a covariate that I know in advance?

If you have a variable that is known for the entire forecast horizon (for example, a month indicator, a scheduled event flag, or an external regressor), you can pass it as a `known_covariate`:

```python
predictor = TimeSeriesPredictor(
    prediction_length=12,
    target="target",
    known_covariates_names=["month", "is_holiday"]
)
predictor.fit(train_data, ...)

# At prediction time, also supply the future covariate values
future_covariates = ...  # TimeSeriesDataFrame with only the covariate columns
predictions = predictor.predict(train_data, known_covariates=future_covariates)
```

Models that support covariates (like Temporal Fusion Transformer) will use them automatically. Others will ignore them. The leaderboard will tell you which approach helped.


---

## What's Next?

This notebook covered the core time series forecasting workflow with AutoGluon. A few directions worth exploring from here:

- **Fine-tune Chronos on your data.** The [AutoGluon Chronos tutorial](https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-chronos.html) shows how to fine-tune when zero-shot is not quite enough.
- **Add covariates.** The [covariates tutorial](https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-indepth.html) walks through how to incorporate known future variables.
- **Go deeper on forecasting methodology.** Hyndman and Athanasopoulos's *Forecasting: Principles and Practice* (free online at [otexts.com/fpp3](https://otexts.com/fpp3)) is the reference text for understanding ETS, ARIMA, and evaluation metrics like MASE.
- **See the Handbook chapter.** The [MIDAS AI in Research Handbook](https://midas-ai-in-research.readthedocs.io) chapter on this notebook covers the concepts in more depth, including guidance on when forecasting is and is not the right frame for your research question.

**Citations:** If you use AutoGluon in your research, please cite:

> Shchur, O., Turkmen, A. C., Erickson, N., et al. (2023). AutoGluon-TimeSeries: AutoML for Probabilistic Time Series Forecasting. *arXiv preprint arXiv:2308.05566*.

> Makridakis, S., Spiliotis, E., & Assimakopoulos, V. (2020). The M4 Competition: 100,000 time series and 61 forecasting methods. *International Journal of Forecasting, 36*(1), 54–74.
